In [ ]:
using Pkg
Pkg.activate(@__DIR__)

In [ ]:
# Auto-develop the parent package if it is not yet in the environment
if !haskey(Pkg.project().dependencies, "TrustRegionRadius")
    @info "Developing TrustRegionRadius from parent directory…"
    Pkg.develop(PackageSpec(path = joinpath(@__DIR__, "..")))
end

In [ ]:
Pkg.instantiate()

In [ ]:
using Revise
using TrustRegionRadius
using CUTEst
using JLD2
using LinearAlgebra
using Printf

In [ ]:
const SOLVER_PARAMS = TRSolverParams(
    η₁ = 0.1,
    η₂ = 0.9,
    Δ₀ = 1.0,
    max_iterations = 10_000,
    tol = 1e-5,
)

In [ ]:
# Factory functions so each run gets a fresh (and for R4 mutable) rule
const RULES = [
    ("R1", () -> R1ClassicalUpdate(0.25, 0.50, 2.0)),
    ("R2", () -> R2StepSizeUpdate(0.25, 0.80, 2.0)),
    ("R3", () -> R3DFOLikeUpdate(0.25, 0.50, 2.0, 1.0)),
    ("R4", () -> R4RelativeGradUpdate(0.25, 2.0,  1.0)),
]

In [ ]:
try
    finalize(nlp) 
catch e
    @error "No model to finalize: $e"
end

In [ ]:
# =============================================================================
# Problem selection
# =============================================================================

@info "Querying CUTEst problem list…"

prob_name = "ROSENBR"
nlp = CUTEstModel(prob_name)

In [ ]:
# Safety check: skip if problem has constraints or wrong size
@show nvar = nlp.meta.nvar
@show ncon = nlp.meta.ncon
if nvar < 2 || nvar > 500 || ncon > 0
    @warn "  $rule_name: skipping $prob_name (nvar=$nvar, ncon=$ncon)"
    finalize(nlp)
    nlp = nothing
end

In [ ]:
rule_number = 3
rule = RULES[rule_number][2]()

In [ ]:
SOLVER_PARAMS

In [ ]:
t0  = time()
out = trust_region_solver(nlp, rule, SOLVER_PARAMS)
elapsed = time() - t0

In [ ]:
out

In [ ]:
out.delta_trajectory